In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
df=pd.read_csv("price.csv")
print(df.head())
print(df.describe())


In [ ]:
print(df['Area'].duplicated())

In [ ]:
df=df[~((df['Price']<100000) & (df['Area']>4000))]
df


In [ ]:
df=df[~(df['Price']>952173) & (df['Area']<1000)]


In [ ]:
df

In [ ]:
df1=df.drop(columns=["Id"])

In [ ]:
relation=df1.corr(numeric_only=True)
print(relation)
df1['Location'].unique()

In [ ]:
dum=pd.get_dummies(df1['Location'],dtype=int,drop_first=True)
dum

In [ ]:
dum2=pd.get_dummies(df1['Condition'],dtype=int,drop_first=True)



In [ ]:
dum2

In [ ]:
df1['Garage']=df1['Garage'].map({
    'Yes':1,
    'No':0
})
df1

In [ ]:
df1 = pd.concat([df1, dum, dum2], axis=1)
df1


In [ ]:
df1.columns

In [ ]:
df2=df1.drop(columns=['Location','Condition'])

In [ ]:
df2

In [ ]:

train=df2.sample(frac=0.8,random_state=0)
test=df2.drop(train.index)



In [ ]:
relation=train.corr(numeric_only=True)


In [ ]:
test
x_test=test.drop(columns=['Price'])
y_test=test['Price']
x_test['Intercept']=1
x_test

In [ ]:
train.columns


In [ ]:
x_train=train.drop(columns=['Price'])

In [ ]:
y_train=train['Price']

In [ ]:
x_train['Intercept']=1

In [ ]:
x_train


In [ ]:
import torch

In [ ]:
x_test

In [ ]:
X_numpy = x_train.to_numpy(dtype='float32')
y_numpy = y_train.to_numpy(dtype='float32')
x_tes=x_test.to_numpy(dtype='float32')
y_tes=y_test.to_numpy(dtype='float32')

In [ ]:
x_torch=torch.from_numpy(X_numpy)
y_torch=torch.from_numpy(y_numpy).reshape(-1,1)
x_tes=torch.from_numpy(x_tes)
y_tes=torch.from_numpy(y_tes)

In [ ]:
print(y_torch.size())
print(x_torch.size())
print(x_tes.size())

In [ ]:
x_t=torch.t(x_torch)
print(x_t)
x_mul=torch.matmul(x_t,x_torch)
print(x_mul)

In [ ]:
x_inv=torch.inverse(x_mul)
print(x_inv)

In [ ]:
fin=torch.matmul(x_inv,x_t)
print(fin)
fin=torch.matmul(fin,y_torch)
print(fin)

In [ ]:
fin
print(x_tes.size())

In [ ]:
y_pred=torch.matmul(x_tes,fin)

In [ ]:
y_pred

In [ ]:
val=(y_tes-y_pred)**2

In [ ]:
val

In [ ]:
mse=1/len(val)*(sum(val))
mse
me=sum(mse)/len(mse)
print(me)

In [ ]:
w = torch.linalg.pinv(x_torch)
w=torch.matmul(w,y_torch)
w


In [ ]:
y_pred=torch.matmul(x_tes,w)
val=(y_tes-y_pred)**2
mse=1/len(val)*(sum(val))
mse
me=sum(mse)/len(mse)
print(me)

In [ ]:
x_torch

In [ ]:
w= torch.linalg.lstsq(x_torch, y_torch, driver='gelsd').solution

In [ ]:
y_pred=torch.matmul(x_tes,w)
val=(y_tes-y_pred)**2
mse=1/len(val)*(sum(val))
mse
me=sum(mse)/len(mse)
print(me)

In [ ]:
weights = torch.linalg.lstsq(x_torch, y_torch, driver='gelsd').solution

y_pred = x_tes @ weights

mse = torch.mean((y_tes - y_pred)**2)
rmse = torch.sqrt(mse)
rmse

In [ ]:
import torch.nn as nn
input_dim = x_torch.shape[1]
model = nn.Linear(input_dim, 1, bias=False)
optimizer = torch.optim.Adam(model.parameters(), lr=10)
criterion = nn.MSELoss()
epochs = 30000
for epoch in range(epochs):
    # Forward pass
    y_pred = model(x_torch)
    loss = criterion(y_pred, y_torch)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 500 == 0:
        current_rmse = torch.sqrt(loss).item()
        print(f'Epoch [{epoch+1}/{epochs}], RMSE: ${current_rmse:,.2f}')

model.eval()
with torch.no_grad():
    test_pred = model(x_tes)
    test_rmse = torch.sqrt(torch.mean((y_tes.reshape(-1,1) - test_pred)**2))
    print(f'\n--- Final Test RMSE: ${test_rmse.item():,.2f} ---')